In [1]:
!pip install onnxruntime -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 79.6 MB/s eta 0:00:00:00:0100:01


In [2]:
!pip install omegaconf -q

In [3]:
import re
import unicodedata
import string
import numpy as np
import pandas as pd
from nltk.tokenize import word_tokenize
from typing import List
import nltk
nltk.download('punkt_tab', quiet=True)

True

In [4]:
from typing import List

import numpy as np
import onnxruntime as ort
from huggingface_hub import hf_hub_download
from omegaconf import OmegaConf
from sentencepiece import SentencePieceProcessor

In [5]:
from tqdm.notebook import tqdm
tqdm.pandas()

In [6]:
DATA_PATH = '/kaggle/input/datasets/arinazamyshevskaya/rlc-prep/sentences_full.csv'

In [7]:
df = pd.read_csv(DATA_PATH)

# Handle legacy sentences.csv columns
if 'status' in df.columns:
    df = df[df['status'] != 'needs correction']
if 'corrected' in df.columns:
    df = df[~df['corrected'].isna()]
for col in ['prev_sent', 'next_sent']:
    if col not in df.columns:
        df[col] = ''
df = df.dropna(subset=['text', 'corrected'])
print(f'Loaded {len(df)} rows')

Loaded 51096 rows


In [8]:
def normalize(text):
    text = unicodedata.normalize('NFC', str(text))
    return re.sub(r'\s+', ' ', text).strip()

df['text'] = df['text'].progress_apply(normalize)
df['corrected'] = df['corrected'].progress_apply(normalize)
df['prev_sent'] = df['prev_sent'].fillna('').apply(normalize)
df['next_sent'] = df['next_sent'].fillna('').apply(normalize)

  0%|          | 0/51096 [00:00<?, ?it/s]

  0%|          | 0/51096 [00:00<?, ?it/s]

In [9]:
from huggingface_hub import hf_hub_download
from omegaconf import OmegaConf
from sentencepiece import SentencePieceProcessor
import onnxruntime as ort

# 1-800-BAD-CODE/xlm-roberta_punctuation_fullstop_truecase
REPO = '1-800-BAD-CODE/xlm-roberta_punctuation_fullstop_truecase'
spe_path = hf_hub_download(repo_id=REPO, filename='sp.model')
onnx_path = hf_hub_download(repo_id=REPO, filename='model.onnx')
config_path = hf_hub_download(repo_id=REPO, filename='config.yaml')

sp_model: SentencePieceProcessor = SentencePieceProcessor(spe_path)
ort_sess: ort.InferenceSession   = ort.InferenceSession(onnx_path)
cfg = OmegaConf.load(config_path)
pre_labels: List[str] = cfg.pre_labels
post_labels: List[str] = cfg.post_labels
null_tok = cfg.get('null_token',   '<NULL>')
acronym_tok = cfg.get('acronym_token','<ACRONYM>')

sp.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

model.onnx:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

config.yaml:   0%|          | 0.00/531 [00:00<?, ?B/s]

In [10]:
MAX_CHUNK = 500  # xlm-roberta uses position offset +2, so N tokens → positions 2..N+1
                 # table has 514 entries (0-513), safe limit: 512 - 2 = 510; using 500 for margin

def _run_chunk(chunk_ids):
    input_ids = [sp_model.bos_id()] + chunk_ids + [sp_model.eos_id()]
    ids_arr = np.array([input_ids])
    pre_p, post_p, cap_p, sbd_p = ort_sess.run(None, {'input_ids': ids_arr})
    pre_p, post_p, cap_p, sbd_p = pre_p[0].tolist(), post_p[0].tolist(), cap_p[0].tolist(), sbd_p[0].tolist()
    output_texts, cur_chars = [], []
    for tok_idx in range(1, len(input_ids) - 1):
        token = sp_model.IdToPiece(input_ids[tok_idx])
        pre_label = pre_labels[pre_p[tok_idx]]
        post_label = post_labels[post_p[tok_idx]]
        if token.startswith('\u2581') and cur_chars:
            cur_chars.append(' ')
        if pre_label != null_tok:
            cur_chars.append(pre_label)
        char_start = 1 if token.startswith('\u2581') else 0
        for ci, ch in enumerate(token[char_start:], start=char_start):
            cur_chars.append(ch.upper() if cap_p[tok_idx][ci] else ch)
            if post_label == acronym_tok:
                cur_chars.append('.')
        if post_label not in (null_tok, acronym_tok):
            cur_chars.append(post_label)
        if sbd_p[tok_idx]:
            output_texts.append(''.join(cur_chars))
            cur_chars.clear()
    if cur_chars:
        output_texts.append(''.join(cur_chars))
    return output_texts


def add_punct(input_text: str) -> str:
    all_ids = sp_model.EncodeAsIds(input_text)
    chunks = [all_ids[i:i + MAX_CHUNK] for i in range(0, max(len(all_ids), 1), MAX_CHUNK)]
    output_texts = []
    for chunk_ids in chunks:
        output_texts.extend(_run_chunk(chunk_ids))
    return ' '.join(output_texts)


def restore_punct(text: str) -> str:
    cleaned = ' '.join(w for w in word_tokenize(str(text)) if w not in string.punctuation)
    if not cleaned.strip():
        return text
    restored = add_punct(cleaned)
    return re.sub(r'\s+', ' ', restored).strip()


In [11]:
df['text'] = df['text'].progress_apply(restore_punct)
df['corrected'] = df['corrected'].progress_apply(restore_punct)
print('Punctuation restored')
df[['text', 'corrected']].head()

  0%|          | 0/51096 [00:00<?, ?it/s]

  0%|          | 0/51096 [00:00<?, ?it/s]

Punctuation restored


,text,corrected
0,Студенты так же борятся за спасение озера Байк...,Студенты также борются за спасение озера Байка...
1,Частные институты действуют по средствам рыноч...,Частные институты действуют посредством рыночн...
2,"...Мы видим на рынке процесс, по средствам кот...","...Мы видим на рынке процесс, посредством кото..."
3,У государства экономические и административные...,Государство принимает экономические и админист...
4,.Основные задачи государственного регулировани...,Основными задачами государственного регулирова...


In [12]:
cyrillic = re.compile(r'[а-яА-ЯёЁ]')
df = df[df['text'].str.contains(cyrillic, na=False)]
df = df[df['corrected'].str.contains(cyrillic, na=False)]
df = df[df['text'] != df['corrected']]
df = df.drop_duplicates(subset=['text', 'corrected'])
df = df.reset_index(drop=True)
print(f'Pairs after filtering: {len(df)}')

Pairs after filtering: 50329


In [13]:
out_cols = ['text', 'corrected', 'prev_sent', 'next_sent']
df[out_cols].to_csv('sentences_preprocessed.csv', index=False)
df[['text', 'corrected']].head()

,text,corrected
0,Студенты так же борятся за спасение озера Байк...,Студенты также борются за спасение озера Байка...
1,Частные институты действуют по средствам рыноч...,Частные институты действуют посредством рыночн...
2,"...Мы видим на рынке процесс, по средствам кот...","...Мы видим на рынке процесс, посредством кото..."
3,У государства экономические и административные...,Государство принимает экономические и админист...
4,.Основные задачи государственного регулировани...,Основными задачами государственного регулирова...
